In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np

# Laden des Digits-Datensatzes
digits = load_digits()
X = digits.images  # Form: (n_samples, 8, 8)
y = digits.target

# Normieren und Umformen der Daten für CNN
X = X / 16.0
X = np.expand_dims(X, axis=1)  # Form: (n_samples, 1, 8, 8)

# Aufteilen in Trainings- und Testdaten
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Umwandeln in PyTorch-Tensoren
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

# Kompaktes CNN als ein einziges Sequential-Modell
net = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1),  # 1x8x8 → 8x8x8
    nn.ReLU(),
    nn.MaxPool2d(2),                            # 8x8x8 → 8x4x4
    nn.Flatten(),                               # 8*4*4 = 128
    nn.Linear(8 * 4 * 4, 32),
    nn.ReLU(),
    nn.Linear(32, 10)                           # 10 Klassen (Ziffern)
)

# Verlustfunktion und Optimierer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.01)

# Training
for epoch in range(20):
    optimizer.zero_grad()
    outputs = net(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()
    print(f'Epoch: {epoch}, Loss: {loss.item():.4f}')

# Testauswertung
net.eval()
with torch.no_grad():
    outputs = net(X_test)
    _, predicted = torch.max(outputs, 1)
    accuracy = accuracy_score(y_test, predicted)
    print('Testgenauigkeit:', accuracy)


Epoch: 0, Loss: 2.3161
Epoch: 1, Loss: 2.2895
Epoch: 2, Loss: 2.2698
Epoch: 3, Loss: 2.2446
Epoch: 4, Loss: 2.2093
Epoch: 5, Loss: 2.1650
Epoch: 6, Loss: 2.1111
Epoch: 7, Loss: 2.0462
Epoch: 8, Loss: 1.9692
Epoch: 9, Loss: 1.8780
Epoch: 10, Loss: 1.7753
Epoch: 11, Loss: 1.6634
Epoch: 12, Loss: 1.5408
Epoch: 13, Loss: 1.4096
Epoch: 14, Loss: 1.2740
Epoch: 15, Loss: 1.1378
Epoch: 16, Loss: 1.0055
Epoch: 17, Loss: 0.8824
Epoch: 18, Loss: 0.7726
Epoch: 19, Loss: 0.6781
Testgenauigkeit: 0.9027777777777778


In [3]:
# print the model architecture
print(net)

# print the number of trainable weights of the model:
total_params = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(f'Total number of trainable parameters: {total_params}')

Sequential(
  (0): Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Flatten(start_dim=1, end_dim=-1)
  (4): Linear(in_features=128, out_features=32, bias=True)
  (5): ReLU()
  (6): Linear(in_features=32, out_features=10, bias=True)
)
Total number of trainable parameters: 4538
